In [1]:
#Tsfnet two stage training
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
# FIX (Bug 1 - Root Cause of "No valid preprocessed data found"):
# Console output said "Scanning ... on Drive D" but the path was pointing to Drive E.
# This mismatch means os.path.exists(sub_folder) returns False for every subject,
# so ALL subjects are skipped silently and total_windows stays 0 -> ValueError.
# ACTION: Update this path to match wherever your preprocessed .npy files actually live.
# Common locations to check:
#   r"D:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive D)
#   r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive E)
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # FIX (Bug 1 diagnostic guard): If PROCESSED_DATA_PATH itself does not exist,
        # every sub_folder check silently fails (os.path.exists returns False) and
        # total_windows stays 0, which raises the cryptic "No valid preprocessed data found"
        # error. This guard catches that case immediately with a clear, actionable message.
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        # self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        # self.eeg_bn1 = nn.BatchNorm1d(32)
        # self.eeg_drop1 = nn.Dropout(dropout)
        # self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        # self.eeg_bn2 = nn.BatchNorm1d(64)
        # self.eeg_drop2 = nn.Dropout(dropout)
        # self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        # self.eeg_fc = nn.Linear(64, 64)
        # CHANGED (TSFNet paper Table 1, EEG branch): EEG input is now the spatially-mapped
        # (16, 16, 600) grid (H, W, T), not (28, 600) channel-space data, so Conv1d over channels no
        # longer applies. Conv3d is used instead, with kernel/stride taken directly from Table 1
        # (kernel 4x4x12, stride 2x2x6 for Conv1; kernel 2x2x6, stride 2x2x2 for Conv2, read as
        # spatial x spatial x temporal). PyTorch's Conv3d expects input (N, C, D, H, W); D is used
        # for the TEMPORAL axis here (D=600 initially) and H, W for the spatial 16x16 grid, so
        # kernel_size/stride/padding below are given as (temporal, spatial_H, spatial_W) tuples.
        # padding is chosen (see chat derivation) to reproduce the paper's "adaptive padding"
        # behavior -- i.e. each conv halves/downsamples proportionally via output=ceil(input/stride)
        # -- which is what makes the Table 1 output dimensions (8x8x100x16, then 4x4x50x32) exact:
        #   Conv1: in (600,16,16) -> out (100,8,8), 16 channels  [Table 1: 8x8x100x16]
        #   Conv2: in (100,8,8)   -> out (50,4,4),  32 channels  [Table 1: 4x4x50x32]
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))  # NEW
        self.eeg_bn1 = nn.BatchNorm3d(16)  # NEW
        self.eeg_drop1 = nn.Dropout(dropout)  # NEW
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))  # NEW
        self.eeg_bn2 = nn.BatchNorm3d(32)  # NEW
        self.eeg_drop2 = nn.Dropout(dropout)  # NEW
        # NEW: GAP is applied ONLY over the temporal axis in forward() (x.mean(dim=2)), NOT via
        # AdaptiveAvgPool1d(1) -- that used to collapse the ENTIRE feature map (channels AND
        # whatever spatial structure existed) down to one scalar per channel. The 4x4 spatial map
        # produced by eeg_conv2 (matching the paper's F^eeg in R^{4x4x32}) is preserved instead, so
        # eeg_pool as a module is no longer needed; there is no self.eeg_pool anymore.
        # self.eeg_fc = nn.Linear(64, 64)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: input is now the flattened (32,4,4) spatial map (512-d), not a pre-pooled 64-d vector

        # fNIRS branch
        # self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        # self.nirs_bn1 = nn.BatchNorm3d(16)
        # self.nirs_drop1 = nn.Dropout(dropout)
        # self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        # self.nirs_bn2 = nn.BatchNorm3d(32)
        # self.nirs_drop2 = nn.Dropout(dropout)
        # self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        # self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        # CHANGED (Run H, 11-segment fNIRS -- paper Table 1 compliance, verified directly against
        # the paper's full text, Sec. 2.3 and Table 1): input is now (batch, 11, 16, 16, 30, 2) --
        # 11 segments (current + subsequent 10, per "each EEG time window was paired with the
        # current and subsequent 10 fNIRS segments... each fNIRS sample had a shape of
        # 11 x 36 x 30 x 2"), each segment 3s long (30 samples @ 10Hz), 2 channels (HbO/HbR).
        # CRITICAL per Table 1: the segment axis (11) is NOT touched by either conv layer -- only
        # the spatial (16x16) and per-segment-internal-time (30) axes are reduced. This is achieved
        # by folding the segment axis into the batch dimension before each conv (so the same
        # Conv3d weights are applied independently, per-segment, exactly like a shared/distributed
        # convolution), then folding it back out afterward. This directly fixes the bug found in a
        # reviewed third-party implementation, which applied a plain Conv3d across all three of
        # (segment, spatial, spatial) jointly, silently shrinking the segment axis from 11 to 2 and
        # requiring a nonsensical repeat+truncate patch downstream.
        # Table 1 states: Conv1 kernel 4x4x6, stride 2x2x2, 16 channels -> output 11x8x8x15x16
        # (segment x H x W x T x channels); Conv2 kernel 2x2x3, stride 2x2x2, 32 channels -> output
        # 11x4x4x8x32. Padding values are NOT stated explicitly in the fetched paper text (only
        # kernel/stride/output dims were) -- the padding below (2,1,1) and (1,0,0) were derived to
        # reproduce exactly those stated output dimensions, not directly quoted from the paper.
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))  # NEW: (T,H,W) order, matches EEG branch's convention; 30->15(T), 16->8(H,W)
        self.nirs_bn1 = nn.BatchNorm3d(16)  # NEW
        self.nirs_drop1 = nn.Dropout(dropout)  # NEW
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))  # NEW: 15->8(T), 8->4(H,W)
        self.nirs_bn2 = nn.BatchNorm3d(32)  # NEW
        self.nirs_drop2 = nn.Dropout(dropout)  # NEW
        # self.nirs_fc = nn.Linear(32, 64)
        # CHANGED: fNIRS's own conv stack now reduces the 16x16 spatial grid to 4x4 (16->8->4),
        # matching the EEG branch's spatial resolution, same as before -- only the input/conv
        # parameters changed, not this downstream sizing.
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)  # CHANGED: (32,) -> (32*4*4,) input, spatial map preserved

        # EFGF
        # self.efgf_eeg = nn.Linear(64, 64)
        # self.efgf_nirs = nn.Linear(64, 64)
        # self.efgf_gate = nn.Linear(64, 1)
        # self.alpha = nn.Parameter(torch.tensor(0.5))
        # CHANGED (TSFNet paper Eq. 2): Phi must be a genuine SPATIAL attention map of shape
        # (h, w, 1, 1) -- i.e. (batch, 1, 4, 4) here -- not a single scalar per sample. Eq. 2 is
        # Phi = alpha*sigmoid(GAP(conv_{h,w,t}(F^fnirs))) + (1-alpha)*sigmoid(GAP(conv_{h,w,t}(F^eeg))).
        # Since eeg_feat_map / nirs_feat_map (defined in forward()) are already GAP'd over time
        # while preserving the (32,4,4) spatial map, "conv_{h,w,t} then GAP-over-time" collapses here
        # to a single small conv that reduces the channel dimension 32->1 while preserving the 4x4
        # spatial map (a 1x1 Conv2d, since no further spatial reduction is needed or intended at this
        # depth) -- this is a documented simplification of the paper's literal two-stage EFGF1/EFGF2
        # (Table 1), consistent with this thesis's single-stage fusion architecture.
        self.efgf_eeg_attn = nn.Conv2d(32, 1, kernel_size=1)   # NEW: replaces efgf_eeg + part of efgf_gate
        self.efgf_nirs_attn = nn.Conv2d(32, 1, kernel_size=1)  # NEW: replaces efgf_nirs + part of efgf_gate
        self.alpha = nn.Parameter(torch.tensor(0.0))  # CHANGED: 0.5 -> 0.0 ("all trainable residual parameters initialized to zero" per paper Sec. 3.6); now used ONLY for Eq.2's blend
        self.gamma = nn.Parameter(torch.tensor(0.0))  # NEW: paper Eq.1 uses a SEPARATE trainable residual parameter gamma (also init 0) for blending F^eeg vs F_fusion; previously self.alpha was incorrectly reused for both Eq.1 and Eq.2's distinct roles
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)  # NEW: projects the spatial fused feature map (Eq.1 output) down to the flat 64-d vector the (unchanged) CAFE/classifier stages below expect

        # CAFE
        # self.pos_encoder = PositionalEncoding(64, max_len=100)
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        self.pos_encoder = PositionalEncoding(64, max_len=11)  # CHANGED: 100 -> 11
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        # NEW (TSFNet paper Sec. 3.3 -- CAFE is bidirectional): the paper's CAFE layer has TWO
        # cross-attention modules ("cross-attention 1": fNIRS attends to fusion, to extract the most
        # relevant fNIRS components aligned with the fusion representation; "cross-attention 2":
        # fusion attends to fNIRS, to enhance spatial characteristics -- this second one is what
        # self.cafe_attn/self.cafe_gate above already implement). cafe_attn2/cafe_gate2 add the
        # missing first direction: nirs_feat as query, fused_feat as key/value.
        self.cafe_attn2 = CrossAttention(64, nhead=4, dropout=dropout)  # NEW: cross-attention 1 (fNIRS <- fusion)
        self.cafe_gate2 = nn.Linear(128, 64)  # NEW: gated residual combination for the enhanced fNIRS representation

        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # EEG branch process
        # x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        # x = self.eeg_drop1(x)
        # x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        # x = self.eeg_drop2(x)
        # eeg_feat = self.eeg_pool(x).view(batch_size, -1)
        # eeg_feat = self.eeg_fc(eeg_feat)
        # CHANGED (EEG spatial mapping): eeg arrives as (batch, 16, 16, 600) = (N, H, W, T) from the
        # dataset (see MMapHybridDataset). Conv3d expects (N, C, D, H, W), so we permute T to the D
        # (depth) position and add a channel dim of 1 (a single interpolated grid, not 28 separate
        # channels anymore).
        eeg = eeg.permute(0, 3, 1, 2).unsqueeze(1)  # NEW: (batch,16,16,600) -> (batch,1,600,16,16)
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))  # NEW: (batch,16,100,8,8)
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))  # NEW: (batch,32,50,4,4)
        x = self.eeg_drop2(x)
        eeg_feat_map = x.mean(dim=2)  # NEW: GAP over TIME axis only (dim=2=D) -> (batch,32,4,4), spatial map preserved (paper's F^eeg in R^{4x4x32})
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d (used by fc_eeg classifier and EFGF)

        # fNIRS branch process
        # y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        # y = self.nirs_drop1(y)
        # y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        # y = self.nirs_drop2(y)
        #
        # y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        # nirs_feat_map = y.mean(dim=2)
        # nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))
        # CHANGED (Run H, 11-segment fNIRS -- Table 1 compliance): nirs arrives as
        # (batch, 11, 16, 16, 30, 2) = (batch, segment, H, W, T, channels) from the dataset. To
        # apply the SAME Conv3d weights to each of the 11 segments independently (i.e. the segment
        # axis must NOT be touched by the conv, per Table 1's stated output dims), the segment axis
        # is folded into the batch dimension before the conv stack, then unfolded afterward.
        n_seg = nirs.size(1)  # NEW: 11
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)  # NEW: (batch,11,16,16,30,2) -> (batch,11,channels=2,T=30,H=16,W=16)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)  # NEW: fold segment into batch -> (batch*11, 2, 30, 16, 16)

        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # NEW: (batch*11, 16, 15, 8, 8)
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))  # NEW: (batch*11, 32, 8, 4, 4)
        y = self.nirs_drop2(y)

        y = y.reshape(batch_size, n_seg, 32, 8, 4, 4)  # NEW: unfold segment axis back out -> (batch, 11, 32, T'=8, H'=4, W'=4)

        # NEW: nirs_feat_map -- a SINGLE spatial map for the whole window (used by EFGF and by the
        # fNIRS classifier head), obtained by averaging over BOTH the segment axis and the
        # per-segment-internal-time axis (mirroring eeg_feat_map's GAP-over-time above).
        nirs_feat_map = y.mean(dim=(1, 3))  # NEW: (batch,11,32,8,4,4) -> mean over dim1(seg) & dim3(T') -> (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))  # CHANGED: flatten (32,4,4)->512, then project to 64-d

        # NEW: the REAL 11-segment sequence for CAFE (replacing the old fabricated/proxy sequence).
        # Average over the per-segment-internal-time and spatial axes, keeping the segment axis
        # intact as the genuine temporal sequence the paper's CAFE layer is designed to attend over.
        y_spatial = y.mean(dim=(3, 4, 5))  # NEW: (batch,11,32,8,4,4) -> mean over T',H',W' -> (batch,11,32)

        # EFGF Block
        # eeg_p = self.efgf_eeg(eeg_feat)
        # nirs_p = self.efgf_nirs(nirs_feat)
        # alpha = torch.clamp(self.alpha, 0, 1)
        # hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        # gate = torch.sigmoid(self.efgf_gate(hybrid))
        # fused_feat = eeg_feat + gate * nirs_feat
        # CHANGED (TSFNet paper Eq. 1-2): Phi is now a genuine SPATIAL attention map (batch,1,4,4),
        # built directly from the (32,4,4) spatial feature maps, not from flattened 64-d vectors.
        # Eq. 2 gives fNIRS weight alpha and EEG weight (1-alpha) -- the previous code had this
        # backwards (alpha weighted EEG); corrected here to match the paper exactly.
        eeg_attn = torch.sigmoid(self.efgf_eeg_attn(eeg_feat_map))    # NEW: (batch,1,4,4)
        nirs_attn = torch.sigmoid(self.efgf_nirs_attn(nirs_feat_map))  # NEW: (batch,1,4,4)
        alpha = torch.clamp(self.alpha, 0, 1)
        Phi = alpha * nirs_attn + (1 - alpha) * eeg_attn  # NEW: Eq.2 -- spatial attention map (batch,1,4,4)

        # NEW (TSFNet paper Eq. 1): Fhat_fusion = gamma*F_eeg + (1-gamma)*F_fusion + Phi (x) F_fusion.
        # This simplified single-stage architecture has no separate parallel "Fusion" CNN branch (see
        # chat discussion of this scope decision), so nirs_feat_map stands in for the paper's
        # F_fusion, exactly as the previous (pre-spatial) code used nirs_feat as the gated term in
        # `fused_feat = eeg_feat + gate * nirs_feat`. Phi broadcasts across the 32 channels (shape
        # (batch,1,4,4) x (batch,32,4,4) -> (batch,32,4,4)) via standard PyTorch broadcasting.
        gamma = torch.clamp(self.gamma, 0, 1)  # NEW
        fused_feat_map = gamma * eeg_feat_map + (1 - gamma) * nirs_feat_map + Phi * nirs_feat_map  # NEW: Eq.1, spatial (batch,32,4,4)
        fused_feat = self.fusion_proj(fused_feat_map.flatten(1))  # NEW: (batch,512) -> (batch,64), for the unchanged CAFE/classifier stages below

        # CAFE Block
        # nirs_seq = y_spatial.transpose(1, 2)
        # CHANGED (Run H, 11-segment fNIRS): y_spatial above is already (batch, 11, 32) -- i.e.
        # (batch, segment/sequence, channels) -- since it was constructed directly in that order
        # above, unlike the old y_spatial (batch, 32, T) which needed a transpose to get the
        # sequence axis into the middle position for nirs_seq_proj/pos_encoder/attention.
        nirs_seq = y_spatial  # CHANGED: no transpose needed, already (batch, 11, 32)
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1)  # cross-attention 2: fusion <- fNIRS (unchanged)
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat

        # NEW (TSFNet paper Sec. 3.3 -- CAFE cross-attention 1): fNIRS attends to fusion features to
        # extract the components most relevant/aligned with the fusion representation. Both nirs_feat
        # and fused_feat are single pooled 64-d vectors here (not the 100-step temporal sequence used
        # above), so each is treated as a length-1 sequence; no positional encoding is applied since
        # a length-1 sequence has nothing to encode a position relative to.
        nirs_query = nirs_feat.unsqueeze(1)   # NEW: (batch,1,64)
        fused_kv = fused_feat.unsqueeze(1)    # NEW: (batch,1,64)
        enhanced_nirs = self.cafe_attn2(nirs_query, fused_kv, fused_kv).squeeze(1)  # NEW: (batch,64)
        gate3 = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat, enhanced_nirs], dim=-1)))  # NEW
        final_nirs_feat = gate3 * enhanced_nirs + (1 - gate3) * nirs_feat  # NEW: this is the paper's "effective fNIRS features" Fe^fnirs

        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        # fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fnirs_out = self.fc_fnirs(self.dropout(final_nirs_feat))  # CHANGED: uses the CAFE-enhanced fNIRS representation, not raw nirs_feat, now that cross-attention 1 actually produces one
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced
        # CHANGED (now that Phi is a genuine spatial attention map, Eq. 12's regularization can use
        # the REAL spatial feature maps instead of the flat-vector stand-ins above): eeg_feat_map
        # (batch,32,4,4) is the actual "average-pooled EEG feature map"; Phi (batch,1,4,4) is
        # broadcast to (batch,32,4,4) to match its element count for PCC (Phi itself multiplies
        # nirs_feat_map with exactly this broadcast in Eq.1 above, so this is consistent with how
        # Phi is actually used, not an arbitrary reshape). For Eq. 13, final_nirs_feat is now the
        # genuine "effective fNIRS features" (CAFE cross-attention 1's gated output), replacing the
        # previous raw-nirs_feat stand-in.
        Phi_broadcast = Phi.expand(-1, 32, -1, -1)  # NEW: (batch,1,4,4) -> (batch,32,4,4), matches eeg_feat_map's element count for PCC
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat_map, Phi_broadcast, final_nirs_feat, enhanced  # CHANGED: eeg_feat_map/Phi_broadcast (genuine spatial EFGF PCC pair), final_nirs_feat (genuine CAFE-enhanced fNIRS)


# NEW (overfitting-fix experiment -- stronger L2 weight decay, per chat history): the previous
# run showed the classic Krogh & Hertz (1992) overfitting signature -- best val accuracy reached
# at epoch 1-2, then train accuracy climbs to 85-90% while val stagnates/declines, indicating the
# ~180k-parameter model is memorizing subject-specific noise rather than learning generalizable
# cognitive-load features, given the unfavorable parameter-to-independent-sample ratio (~259:1)
# in this subject-independent 3-class n-back setup.
#
# Krogh, A., & Hertz, J. A. (1992). "A simple weight decay can improve generalization."
# Advances in Neural Information Processing Systems (NeurIPS), 4, 950-957.
# -- shows analytically that L2 weight decay directly shrinks the effective weight-space volume
#    the optimizer can reach, reducing variance (overfitting) WITHOUT changing the network's
#    representational capacity (unlike pruning or reducing channel counts, which was already
#    tried in this project and found to be mostly noise-level -- see project report Section 4).
#
# This project's code already uses weight_decay=2e-2 (a 40x increase over the original 5e-4),
# confirming this direction was already identified as helpful; the natural next step -- exactly
# as recommended in the prior chat analysis -- is NOT to introduce a new regularization concept,
# but to test whether an EVEN STRONGER L2 penalty further improves generalization, following
# Krogh & Hertz's finding that there is a task-dependent optimal weight-decay coefficient (too
# small under-regularizes, too large under-fits) that must be found empirically, i.e. via a small
# grid search rather than a single guess.
#
# WEIGHT_DECAY_GRID lists the current baseline (2e-2) alongside two stronger candidates (5e-2,
# 1e-1) suggested in the prior analysis. Everything else (seed, data split, architecture,
# optimizer type, learning rate, loss weights, scheduler, early-stopping patience) is held
# EXACTLY fixed across the three runs below, so any difference in outcome can be attributed
# specifically to the weight_decay coefficient -- following the same single-variable-isolation
# discipline used throughout this project (e.g. Run C's bundled-filter mistake, which the project
# report explicitly flags as a methodological lesson).
WEIGHT_DECAY_GRID = [2e-2, 5e-2, 1e-1]  # NEW: baseline + two stronger candidates per Krogh & Hertz 1992

LAMBDA_EEG = 0.2     # Eq. 6 lambda_eeg -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_FNIRS = 0.2   # Eq. 6 lambda_fnirs -- the paper's actual reported optimal value (Sec. 3.6)
LAMBDA_EFGF_REG = 1.0   # Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
LAMBDA_CAFE_REG = 1.0   # Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

# TWO-STAGE TRAINING (paper Sec. 3.6):
# Stage 1: train on 80% of train set, early-stop on 20% val, patience=50, max=300 epochs.
#          Best checkpoint (highest val acc) is restored at end of stage 1.
# Stage 2: retrain on FULL train set (no val split), stop when train loss drops below
#          the best stage-1 train loss achieved, max=200 epochs.
# CHANGED: single EPOCHS=100 / EARLY_STOP_PATIENCE=15 replaced by per-stage constants below.
STAGE1_MAX_EPOCHS = 300   # CHANGED: was EPOCHS=100 -- paper Sec. 3.6 says max 300 for stage 1
STAGE1_PATIENCE   = 50    # CHANGED: was EARLY_STOP_PATIENCE=15 -- paper says patience 50
STAGE2_MAX_EPOCHS = 200   # NEW: paper Sec. 3.6 says max 200 for stage 2
STAGE1_VAL_SPLIT  = 0.2   # NEW: paper "4:1 ratio" -> 20% of train set used as val in stage 1


# NEW: the full train+evaluate pipeline (previously the top-level "MAIN PIPELINE" block) is now
# wrapped in a function parameterized by weight_decay, so it can be run once per grid-search
# candidate with everything else held fixed. The body is otherwise UNCHANGED from the original
# script -- no architecture, loss, optimizer type, learning rate, scheduler, or data-loading logic
# was touched, per the instruction to implement only the weight-decay experiment.
def run_training(weight_decay, device):  # NEW
    print("\n" + "#" * 70)
    print(f"# WEIGHT_DECAY = {weight_decay}")
    print("#" * 70)

    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    # NOTE: val_dataset removed -- stage-1 internal validation is a 80/20 split of train_dataset.
    # VAL_SUBS subjects are no longer used during training; they remain as a held-out set
    # that can be added to test evaluation if desired.
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dataloaders
    # NOTE: val_loader and test_loader are for the FINAL test evaluation only.
    # Stage-1 internal validation uses a 80/20 split of train_dataset created inside the training block.
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer (stage 1) -- stage 2 creates its own optimizer2 below
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.CrossEntropyLoss()

    ckpt_path = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_wd{weight_decay}.pth')

    # ------------------------------------------------------------------ #
    # STAGE 1: train on 80% subset, early-stop on 20% val (paper Sec 3.6)
    # ------------------------------------------------------------------ #
    # Split the train indices 4:1 for stage-1 internal val (paper: "randomly split into
    # a first-stage training subset and a validation set in a 4:1 ratio").
    from torch.utils.data import Subset
    n_train_total = len(train_dataset)
    n_s1_val = int(n_train_total * STAGE1_VAL_SPLIT)
    s1_indices = list(range(n_train_total))
    random.shuffle(s1_indices)
    s1_train_idx = s1_indices[n_s1_val:]   # 80%
    s1_val_idx   = s1_indices[:n_s1_val]   # 20%

    s1_train_loader = DataLoader(Subset(train_dataset, s1_train_idx),
                                 batch_size=64, shuffle=True,  num_workers=0)
    s1_val_loader   = DataLoader(Subset(train_dataset, s1_val_idx),
                                 batch_size=64, shuffle=False, num_workers=0)

    print(f"STAGE 1 — training on {len(s1_train_idx)} samples, "
          f"val on {len(s1_val_idx)} samples | max {STAGE1_MAX_EPOCHS} epochs | patience {STAGE1_PATIENCE}")

    best_val_acc     = 0.0
    epochs_no_improve = 0
    stage1_best_train_loss = float('inf')   # will be used as stage-2 stopping criterion
    train_acc        = 0.0                  # kept in scope for overfitting_gap below

    for epoch in range(1, STAGE1_MAX_EPOCHS + 1):
        model.train()
        train_loss, train_class_loss, train_efgf_loss, train_cafe_loss, train_correct = 0.0, 0.0, 0.0, 0.0, 0

        for eeg_b, nirs_b, lbl_b in s1_train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss       += loss.item()
            train_class_loss += L_class.item()
            train_efgf_loss  += L_efgf.item()
            train_cafe_loss  += L_cafe.item()
            train_correct    += (pred.argmax(1) == lbl_b).sum().item()

        train_acc      = train_correct / len(s1_train_loader.dataset)
        avg_train_loss = train_loss / len(s1_train_loader)

        # stage-1 validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in s1_val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
        val_acc = val_correct / len(s1_val_loader.dataset)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc           = val_acc
            stage1_best_train_loss = avg_train_loss   # snapshot train loss at best val checkpoint
            torch.save(model.state_dict(), ckpt_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epoch % 5 == 0 or epoch == 1:
            n_batches = len(s1_train_loader)
            print(f"[S1] Epoch {epoch:3d}/{STAGE1_MAX_EPOCHS} | Loss: {avg_train_loss:.4f} | "
                  f"Class: {train_class_loss/n_batches:.4f} | "
                  f"EFGF: {train_efgf_loss/n_batches:.4f} | "
                  f"CAFE: {train_cafe_loss/n_batches:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        if epochs_no_improve >= STAGE1_PATIENCE:
            print(f"\n[S1] Early stopping at epoch {epoch} "
                  f"(no val improvement for {STAGE1_PATIENCE} epochs). "
                  f"Best Val: {best_val_acc*100:.2f}%")
            break

    # Restore best stage-1 weights before stage 2
    model.load_state_dict(torch.load(ckpt_path))
    print(f"\n[S1] Best checkpoint restored. Stage-1 reference train loss: {stage1_best_train_loss:.4f}")

    # ------------------------------------------------------------------ #
    # STAGE 2: retrain on FULL train set, stop when loss < stage-1 best  #
    # (paper Sec 3.6: "This stage was stopped when the training loss      #
    # dropped below that of the first stage")                             #
    # ------------------------------------------------------------------ #
    print(f"\nSTAGE 2 — retraining on full {len(train_dataset)} samples | "
          f"max {STAGE2_MAX_EPOCHS} epochs | target loss < {stage1_best_train_loss:.4f}")

    # Fresh optimizer for stage 2 (same hyper-params) so LR schedule resets cleanly
    optimizer2  = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=weight_decay)
    scheduler2  = optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min', factor=0.5, patience=5)

    for epoch in range(1, STAGE2_MAX_EPOCHS + 1):
        model.train()
        s2_loss, s2_correct = 0.0, 0
        s2_class_loss = s2_efgf_loss = s2_cafe_loss = 0.0

        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            optimizer2.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)

            L_pred   = criterion(pred,      lbl_b)
            L_eeg    = criterion(eeg_out,   lbl_b)
            L_fnirs  = criterion(fnirs_out, lbl_b)
            L_fusion = criterion(fusion_out,lbl_b)
            L_class  = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion
            L_efgf   = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)
            L_cafe   = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)
            loss     = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer2.step()

            s2_loss       += loss.item()
            s2_class_loss += L_class.item()
            s2_efgf_loss  += L_efgf.item()
            s2_cafe_loss  += L_cafe.item()
            s2_correct    += (pred.argmax(1) == lbl_b).sum().item()

        avg_s2_loss = s2_loss / len(train_loader)
        train_acc   = s2_correct / len(train_loader.dataset)
        scheduler2.step(avg_s2_loss)

        if epoch % 5 == 0 or epoch == 1:
            n_b = len(train_loader)
            print(f"[S2] Epoch {epoch:3d}/{STAGE2_MAX_EPOCHS} | Loss: {avg_s2_loss:.4f} | "
                  f"Class: {s2_class_loss/n_b:.4f} | "
                  f"EFGF: {s2_efgf_loss/n_b:.4f} | "
                  f"CAFE: {s2_cafe_loss/n_b:.4f} | "
                  f"Train Acc: {train_acc*100:.2f}% | Target loss: {stage1_best_train_loss:.4f}")

        # Stop criterion: train loss has dropped below stage-1 best (paper Sec. 3.6)
        if avg_s2_loss < stage1_best_train_loss:
            print(f"\n[S2] Stopping at epoch {epoch}: train loss {avg_s2_loss:.4f} "
                  f"< stage-1 target {stage1_best_train_loss:.4f}")
            torch.save(model.state_dict(), ckpt_path)   # save stage-2 final weights
            break
    else:
        # Reached max epochs without meeting loss criterion -- save anyway
        torch.save(model.state_dict(), ckpt_path)
        print(f"[S2] Reached max epochs ({STAGE2_MAX_EPOCHS}) without meeting loss target. Saving final weights.")

    # FINAL MODEL TEST & EVALUATION
    print(f"\n RUNNING FINAL TEST EVALUATION (weight_decay={weight_decay})")
    model.load_state_dict(torch.load(ckpt_path))
    model.eval()

    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)
            all_preds_final.extend(pred.argmax(1).cpu().numpy())
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']

    test_acc = accuracy_score(all_true, all_preds_final) * 100
    kappa_final = cohen_kappa_score(all_true, all_preds_final)
    print(f"\nOverall Model Test Accuracy (final fused prediction): {test_acc:.2f}%")
    print(f"Kappa Score (final fused prediction): {kappa_final:.4f}")
    print("\nClassification Report (final fused prediction):")
    print(classification_report(all_true, all_preds_final, target_names=class_names))

    cm = confusion_matrix(all_true, all_preds_final)
    print("\nConfusion Matrix (rows=true, cols=predicted):")
    header = "        " + "".join([f"{c:>10}" for c in class_names])
    print(header)
    for i, row in enumerate(cm):
        row_str = "".join([f"{v:>10}" for v in row])
        print(f"{class_names[i]:>8}{row_str}")

    eeg_acc = accuracy_score(all_true, all_preds_eeg) * 100
    fnirs_acc = accuracy_score(all_true, all_preds_fnirs) * 100
    fusion_acc = accuracy_score(all_true, all_preds_fusion) * 100
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")
    print(f"  EEG-branch-only accuracy:          {eeg_acc:.2f}%")
    print(f"  fNIRS-branch-only accuracy:         {fnirs_acc:.2f}%")
    print(f"  EFGF+CAFE fused-feature accuracy:   {fusion_acc:.2f}%")
    print(f"  Final weighted-combination accuracy: {test_acc:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")

    with torch.no_grad():
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()
    print(f"\nLearned final-combination weights (after softmax): "
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")

    # NEW: this run's train/val curve summary, to directly diagnose whether the overfitting
    # signature (best val at epoch 1-2, train acc climbing well past val) is reduced at this
    # weight_decay value -- the exact question Krogh & Hertz's theory predicts this experiment
    # should answer.
    overfit_gap = (train_acc * 100) - test_acc  # NEW: simple train-vs-test accuracy gap, a direct overfitting indicator
    print(f"\nOverfitting indicator (final-epoch train_acc - test_acc): {overfit_gap:+.2f} percentage points")

    results = {
        'weight_decay': weight_decay,
        'test_accuracy_final': test_acc,
        'kappa_final': kappa_final,
        'confusion_matrix_final': cm.tolist(),
        'ablation': {
            'eeg_only': eeg_acc,
            'fnirs_only': fnirs_acc,
            'fusion_only': fusion_acc,
            'final_weighted': test_acc,
        },
        'learned_weights': {
            'w_eeg': float(learned_weights[0]),
            'w_fnirs': float(learned_weights[1]),
            'w_fusion': float(learned_weights[2]),
        },
        'best_val_acc': best_val_acc * 100,
        'final_train_acc': train_acc * 100,
        'overfitting_gap': overfit_gap,
    }
    results_path = os.path.join(PROCESSED_DATA_PATH, f'results_wd{weight_decay}.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    train_dataset.cleanup()
    test_dataset.cleanup()

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


# MAIN PIPELINE -- runs the weight-decay grid search
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on Device: {device}")
    print(f"Weight decay grid to test: {WEIGHT_DECAY_GRID}")

    all_grid_results = []  # NEW
    for wd in WEIGHT_DECAY_GRID:  # NEW: loop over the grid-search candidates
        result = run_training(wd, device)
        all_grid_results.append(result)

    # NEW: summary table across the whole weight_decay grid, so the best candidate (by test
    # accuracy AND by the smallest train-test overfitting gap) can be identified at a glance,
    # per Krogh & Hertz's point that the optimal decay coefficient must be found empirically.
    print("\n" + "=" * 70)
    print("WEIGHT DECAY GRID SEARCH SUMMARY")
    print("=" * 70)
    print(f"{'weight_decay':>14} | {'test_acc':>9} | {'best_val_acc':>13} | {'overfit_gap':>12}")
    for r in all_grid_results:
        print(f"{r['weight_decay']:>14} | {r['test_accuracy_final']:>8.2f}% | "
              f"{r['best_val_acc']:>12.2f}% | {r['overfitting_gap']:>+11.2f}pp")

    best_by_test_acc = max(all_grid_results, key=lambda r: r['test_accuracy_final'])
    best_by_gap = min(all_grid_results, key=lambda r: r['overfitting_gap'])
    print(f"\nBest weight_decay by test accuracy: {best_by_test_acc['weight_decay']} "
          f"({best_by_test_acc['test_accuracy_final']:.2f}%)")
    print(f"Best weight_decay by smallest overfitting gap: {best_by_gap['weight_decay']} "
          f"({best_by_gap['overfitting_gap']:+.2f}pp)")

    summary_path = os.path.join(PROCESSED_DATA_PATH, 'weight_decay_grid_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(all_grid_results, f, indent=2)
    print(f"\nFull grid-search summary saved to: {summary_path}")

    print("\nCleanup completed. Temporary disk files removed.")

Running on Device: cuda
Weight decay grid to test: [0.02, 0.05, 0.1]

######################################################################
# WEIGHT_DECAY = 0.02
######################################################################
Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Test_Set - 5 subjects on Drive D...
VP008 verif